# 🎯 核心依赖包

In [1]:
# 📦 安装核心依赖包
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 #colab一般都有
!pip install numpy matplotlib pandas
!pip install GPUtil  # GPU监控
# 🚀 可选：增强功能包
!pip install tensorboard  # 训练可视化
!pip install tqdm         # 进度条美化
!pip install seaborn      # 高级绘图

/bin/bash: pip: command not found
/bin/bash: pip: command not found
/bin/bash: pip: command not found
/bin/bash: pip: command not found
/bin/bash: pip: command not found
/bin/bash: pip: command not found


# 算力设备信息检查

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# === 新增：定义运算设备 ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用设备: {device}")

# 1. 检查 PyTorch 是否能看到 GPU
cuda_available = torch.cuda.is_available()
print(f"1. PyTorch 是否检测到 GPU: {cuda_available}")

if cuda_available:
    print(f"   使用的 GPU 型号: {torch.cuda.get_device_name(0)}")
else:
    print("   ⚠️ 当前环境只有 CPU 可用。可能是 CUDA 没装好，或者环境配置不对。")

# 2. 检查你的模型究竟在哪跑（前提是你已经实例化了 agent）
# 假设你之前跑了 agent = TD3(...)
try:
    # 查看 Actor 网络第一层参数所在的设备
    device_used = next(agent.actor.parameters()).device
    print(f"\n2. 你的 Actor 网络当前运行在: {device_used}")
    if device_used.type == 'cpu':
        print("   ⚠️ 结果：模型在 CPU 上。你需要用 .to('cuda') 把模型搬到 GPU 上。")
    else:
        print("   ✅ 结果：模型已经在 GPU 上了！")
except NameError:
    print("\n2. 请在实例化 agent (TD3) 之后运行此段代码来检查模型位置。")

当前使用设备: cuda
1. PyTorch 是否检测到 GPU: True
   使用的 GPU 型号: NVIDIA GeForce RTX 3090

2. 请在实例化 agent (TD3) 之后运行此段代码来检查模型位置。


In [6]:
import GPUtil

def get_nvidia_gpu_info():
    """
    获取NVIDIA显卡信息（型号、显存、使用率）
    :return: 列表，每个元素为显卡的详细信息
    """
    gpus = GPUtil.getGPUs()
    if not gpus:
        return None
    gpu_info = []
    for gpu in gpus:
        gpu_info.append({
            "显卡ID": gpu.id,
            "型号": gpu.name,
            "总显存(GB)": round(gpu.memoryTotal, 2),
            "已用显存(GB)": round(gpu.memoryUsed, 2),
            "空闲显存(GB)": round(gpu.memoryFree, 2),
            "显卡使用率(%)": gpu.load * 100,
            "温度(℃)": gpu.temperature
        })
    return gpu_info

# 测试
if __name__ == "__main__":
    print("=== NVIDIA显卡信息 ===")
    nvidia_gpu = get_nvidia_gpu_info()
    if nvidia_gpu:
        for idx, gpu in enumerate(nvidia_gpu, 1):
            print(f"\n显卡{idx}:")
            for key, value in gpu.items():
                print(f"  {key}: {value}")
    else:
        print("未检测到NVIDIA显卡")


=== NVIDIA显卡信息 ===

显卡1:
  显卡ID: 0
  型号: NVIDIA GeForce RTX 3090
  总显存(GB): 24576.0
  已用显存(GB): 1014.0
  空闲显存(GB): 23244.0
  显卡使用率(%): 11.0
  温度(℃): 52.0


In [7]:
#td3.py
import copy
import numpy as np
import torch
import torch.nn.functional as F  # 添加导入
import torch.optim as optim

#  from model import Actor, Critic

class TD3:
    def __init__(self, state_dim, action_dim, max_action):
        self.max_action = max_action
        self.actor = Actor(state_dim, action_dim, max_action).to(device)
        self.actor_target = copy.deepcopy(self.actor)

        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = copy.deepcopy(self.critic)

        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=3e-4)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=3e-4)

        self.discount = 0.99
        self.tau = 0.005
        self.policy_noise = 0.2
        self.noise_clip = 0.5
        self.policy_freq = 2
        self.total_it = 0  # 初始化迭代计数器

#     def select_action(self, state):
#         # 1. 把 CPU 上的 numpy 状态变成 Tensor，搬到 GPU 上
#         state = torch.FloatTensor(state.reshape(1, -1)).to(device)
        
#         # 2. 模型在 GPU 里算完动作后，【先 .cpu() 搬回来】，再去 .numpy()
#         # === 核心修改在这里：加上 .cpu() ===
#         action = self.actor(state).cpu().data.numpy().flatten() 
        
#         # 3. 后面的代码保持不变
#         noise = np.random.normal(0, self.policy_noise, size=action.shape)
#         return np.clip(action + noise, -self.max_action, self.max_action)
# ======AI对目标策略平滑噪声做了衰减，我这里把目标策略平滑噪声写死，因为论文原文的意思是目标策略平滑噪声不能变，【修改】 ======
    def select_action(self, state, exploration_noise=0.0): # <--- 增加一个外部噪声参数
        # 1. 把 CPU 上的 numpy 状态变成 Tensor，搬到 GPU 上
        state = torch.FloatTensor(state.reshape(1, -1)).to(device)
        
        # 2. 模型在 GPU 里算完动作后，【先 .cpu() 搬回来】，再去 .numpy()
        action = self.actor(state).cpu().data.numpy().flatten() 
        
        # 3. 如果传入了探索噪声，就加上它；否则输出纯净的确定性动作
        if exploration_noise != 0.0:
            noise = np.random.normal(0, exploration_noise, size=action.shape)
            action = action + noise
            
        return np.clip(action, -self.max_action, self.max_action)

    def update(self, replay_buffer, batch_size=100):
        self.total_it += 1

        # 从经验回放中采样
        states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

        # Critic网络更新
        with torch.no_grad():
            noise = (torch.randn_like(actions) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
            next_actions = (self.actor_target(next_states) + noise).clamp(-self.max_action, self.max_action)

            target_q1, target_q2 = self.critic_target(next_states, next_actions)
            target_q = torch.min(target_q1, target_q2)
            target_q = rewards + (1 - dones) * self.discount * target_q

        current_q1, current_q2 = self.critic(states, actions)
        critic_loss = F.mse_loss(current_q1, target_q) + F.mse_loss(current_q2, target_q)

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

        # 延迟策略更新
        if self.total_it % self.policy_freq == 0:
            # 修复: 正确获取Q值
            q1, _ = self.critic(states, self.actor(states))
            actor_loss = -q1.mean()

            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            # 更新目标网络
            for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

            for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

# ReplayBuffer 保持不变

class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.capacity = capacity
        self.buffer = []
        self.position = 0

    def add(self, state, action, reward, next_state, done):
        if len(self.buffer) < self.capacity:
            self.buffer.append(None)
        self.buffer[self.position] = (state, action, reward, next_state, done)
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size)
        samples = [self.buffer[i] for i in indices]

        # 将列表转换为numpy数组后再转为Tensor
        states = np.array([s[0] for s in samples])
        actions = np.array([s[1] for s in samples])
        rewards = np.array([s[2] for s in samples])
        next_states = np.array([s[3] for s in samples])
        dones = np.array([s[4] for s in samples])

        # return (
        #     torch.FloatTensor(states),
        #     torch.FloatTensor(actions),
        #     torch.FloatTensor(rewards),
        #     torch.FloatTensor(next_states),
        #     torch.FloatTensor(dones)
        # )
        return (
            torch.FloatTensor(states).to(device),
            torch.FloatTensor(actions).to(device),
            torch.FloatTensor(rewards).unsqueeze(1).to(device),  # <--- 如果在算力机器就要增加to(device)在最后增加一个维度变为 [batch, 1]
            torch.FloatTensor(next_states).to(device),
            torch.FloatTensor(dones).unsqueeze(1).to(device)     # <--- 增加这行：同理
        )


    def __len__(self):
        return len(self.buffer)

In [8]:
#model.py
import torch
import torch.nn as nn

class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),  # 输入层改为6维？
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim),
            nn.Tanh()
        )
        self.max_action = max_action

    def forward(self, state):
        return self.net(state) * self.max_action

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.q1 = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

        self.q2 = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, state, action):
        sa = torch.cat([state, action], 1)
        return self.q1(sa), self.q2(sa)

In [9]:
#environment.py
import numpy as np

class DroneEnv:
    """
    无人机路径规划仿真环境
    状态空间：6 or 14维 [x,y,z, dx,dy,dz] (当前位置 + 目标方向向量)
    动作空间：3维 [vx,vy,vz] (三维速度向量)
    """

    def __init__(self):
        # 环境维度参数
        self.action_dim = 3  # 动作空间维度（三维速度）
        # self.state_dim = 6   # 原来的状态空间维度（3D位置 + 3D目标方向）
        # 【修改状态维度】：原来的 6维 + 8条雷达射线 = 14维
        self.state_dim = 14
        self.step_count = 0  # 当前步数计数器
        self.max_step = 200  # 单回合最大步数
        self.prev_distance = 0.0 # 用于记录上一步的距离
        # ================= 新增：雷达射线方向初始化 =================
        # 定义 8 个探测方向 (三维空间对角线的8个方向)
        dirs = np.array([
            [1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1],
            [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]
        ], dtype=np.float32)
        # 将方向向量归一化（长度变成1）
        self.ray_dirs = dirs / np.linalg.norm(dirs, axis=1, keepdims=True)
        self.max_ray_length = 5.0  # 雷达最大探测距离为 5 米
        # ==========================================================
    def _get_lidar_data(self):
        """
        计算8条射线到最近障碍物的距离。
        返回：长度为8的数组，数值经过归一化 (0~1)。1表示安全(没扫到东西)，0表示贴脸。
        """
        distances = np.full(len(self.ray_dirs), self.max_ray_length)

        for i, ray in enumerate(self.ray_dirs):
            min_t = self.max_ray_length

            for obs in self.obstacles:
                # 射线与球体求交点的数学计算 (一元二次方程)
                # 射线公式: P = origin + t * ray
                # 球体公式: ||P - center||^2 = radius^2
                oc = self.position - obs['pos']
                b = 2.0 * np.dot(ray, oc)
                c = np.dot(oc, oc) - obs['radius']**2

                discriminant = b**2 - 4*c  # 判别式 (b^2 - 4ac, a=1因为ray已归一化)

                if discriminant > 0:
                    # 有交点，计算距离 t
                    t1 = (-b - np.sqrt(discriminant)) / 2.0
                    t2 = (-b + np.sqrt(discriminant)) / 2.0

                    # 取大于0且最小的 t (即射线前方最近的交点)
                    if 0 < t1 < min_t:
                        min_t = t1
                    elif 0 < t2 < min_t:
                        min_t = t2

            distances[i] = min_t

        # 归一化测距数据：距离 / 最大探测距离
        return distances / self.max_ray_length
    def reset(self):
        self.step_count = 0
        # 1. 生成出生点和目标点 (保持之前的逻辑)
        self.position = np.random.uniform(-5, 5, size=3)
        self.target = self.position + np.random.uniform(-8, 8, size=3)
        self.target = np.clip(self.target, -14, 14)
        self.target[2] = np.abs(self.target[2]) + 1.0

        self.prev_distance = np.linalg.norm(self.target - self.position)

        # ================= 新增：障碍物生成逻辑 =================
        self.obstacles = []
        num_obstacles = 2  # 前期训练建议先放 2 个障碍物

        for _ in range(num_obstacles):
            # 在无人机和目标点的连线之间，随机找一个比例点 (0.3 到 0.7 之间)
            alpha = np.random.uniform(0.3, 0.7)
            obs_pos = self.position + alpha * (self.target - self.position)

            # 给这个点加一点随机偏移，让障碍物不至于完全挡死直线路径
            obs_pos += np.random.uniform(-1.5, 1.5, size=3)

            # 随机生成障碍物的半径 (0.5米 到 1.5米)
            radius = np.random.uniform(0.5, 1.5)

            # 保存到环境的障碍物列表中
            self.obstacles.append({
                'pos': obs_pos,
                'radius': radius
            })
        # =======================================================

        return self._get_state()

    def _get_state(self):
        """
        生成当前状态向量并进行归一化
        """
        direction = self.target - self.position
        # 将位置和方向都归一化到约 [-1, 1] 范围内 (最大边界为15)
        norm_position = self.position / 15.0
        norm_direction = direction / 15.0
        # 获取雷达数据
        lidar_data = self._get_lidar_data()

        # 【状态拼接】：现在的状态包含了 [位置(3), 目标方向(3), 雷达测距(8)]
        return np.concatenate([norm_position, norm_direction, lidar_data])

    def step(self, action):
        # 1. 物理模拟 (保持推力)
        self.position += action * 0.3
        self.step_count += 1

        # 计算当前到目标的距离
        target_distance = np.linalg.norm(self.target - self.position)

        done = False
        reward = 0.0

        # ================= 新增：碰撞检测与避障惩罚 =================
        collision = False
        min_obs_dist = float('inf')  # 记录离得最近的障碍物距离

        for obs in self.obstacles:
            # 计算无人机中心到障碍物中心的距离
            dist_to_obs = np.linalg.norm(self.position - obs['pos'])
            min_obs_dist = min(min_obs_dist, dist_to_obs)

            # 假设无人机自身半径为 0.2 米
            if dist_to_obs < (obs['radius'] + 0.2):
                collision = True
                break  # 只要撞到一个就算撞毁

        if collision:
            reward = -200.0  # 极度严厉的坠机惩罚
            done = True
            return self._get_state(), reward, done, {}
        # ==========================================================

        # 2. 到达目标检测
        if target_distance < 1.5:
            reward = 100.0  # 成功奖励
            done = True

        # 3. 边界碰撞检测 (越界)
        elif np.any(np.abs(self.position) > 15):
            reward = -50.0  # 撞墙惩罚
            done = True

        # 4. 正常飞行过程中的引导与避障奖励
        else:
            # 势能奖励：鼓励向目标飞
            progress_reward = (self.prev_distance - target_distance) * 10.0

            # 动作平滑惩罚：避免乱抖
            action_penalty = 0.05 * np.linalg.norm(action)

            # 【新增：安全距离斥力惩罚】
            # 如果离障碍物太近（比如不到障碍物半径 + 1米），给一个持续的负反馈逼它绕开
            repulsion_penalty = 0.0
            for obs in self.obstacles:
                dist_to_obs = np.linalg.norm(self.position - obs['pos'])
                safe_margin = obs['radius'] + 1.0
                if dist_to_obs < safe_margin:
                    # 离得越近，惩罚越大
                    repulsion_penalty += 2.0 * (safe_margin - dist_to_obs)

            # 综合本步的总奖励
            reward = progress_reward - action_penalty - repulsion_penalty

            # 超时检测
            if self.step_count >= self.max_step:
                done = True

        # 更新历史距离
        self.prev_distance = target_distance

        return self._get_state(), reward, done, {}

In [10]:
#train.py
# 导入必要的模块
# from environment import DroneEnv  # 无人机仿真环境
# from td3 import TD3, ReplayBuffer  # 强化学习算法及经验回放池
import numpy as np
import torch  # 深度学习框架
import csv  # 新增导入
import os
import argparse
import os
from datetime import datetime
from pathlib import Path

# 初始噪声
current_noise = 0.2
min_noise = 0.01
decay_rate = 0.999
# 初始化无人机训练环境
env = DroneEnv()
state_dim = env.state_dim  # 状态空间维度（位置+目标方向）
action_dim = env.action_dim  # 动作空间维度（三维速度）
max_action = 1.0  # 动作取值范围[-1,1]

# 创建经验回放池（存储转移样本）
replay_buffer = ReplayBuffer(capacity=100000)

# 初始化TD3算法代理
agent = TD3(state_dim, action_dim, max_action)
max_episodes = 5000  # 最大训练回合数

# 生成唯一时间戳用于区分不同运行结果
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")



# 解析命令行参数 (修复 Jupyter/Colab 兼容性问题)
parser = argparse.ArgumentParser()
parser.add_argument('--output_dir', default='./results', help='Directory to save training results')

# 修复A: 在 Colab/Notebook 中运行时，使用 args=[] 忽略系统参数
# 或者如果你想在命令行中运行，可以使用 args = parser.parse_args()
import sys
if 'ipykernel' in sys.modules:
    args = parser.parse_args(args=['--output_dir', './results_drone_Fix_noise']) # 在Colab中强制指定路径
else:
    args = parser.parse_args()

os.makedirs(args.output_dir, exist_ok=True)
print(f"[DEBUG] Output directory: {args.output_dir}")
print(f"[DEBUG] Directory exists: {os.path.isdir(args.output_dir)}")

# 生成唯一时间戳用于区分不同运行结果 (修复B: 确保 datetime 已导入)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_file = os.path.join(args.output_dir, f'training_log_{timestamp}.csv')

# 在训练循环前初始化记录文件
with open(log_file, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['episode', 'reward', 'steps'])

[DEBUG] Output directory: ./results_drone_Fix_noise
[DEBUG] Directory exists: True


In [ ]:
# 开始训练循环
for episode in range(max_episodes):
    # 重置环境获取初始状态
    state = env.reset()
    episode_reward = 0  # 本回合累计奖励

    # 单回合最大步长控制
    for t in range(200):
        # 临时覆盖 agent 的 policy_noise 属性
        # agent.policy_noise = current_noise
        # action = agent.select_action(state)
        # 【修改2】将衰减的探索噪声作为参数传给 select_action
        action = agent.select_action(state, exploration_noise=current_noise)

        # 执行动作并获取环境反馈
        next_state, reward, done, _ = env.step(action)

        # 存储转移样本到经验池
        replay_buffer.add(state, action, reward, next_state, done)

        # 当经验池足够时更新网络参数
        if len(replay_buffer) > 1000:
            agent.update(replay_buffer)

        # 状态转移并累计奖励
        state = next_state
        episode_reward += reward

        # 提前终止条件检查
        if done:
            break
    current_noise = max(min_noise, current_noise * decay_rate)
    # 输出训练进度
    print(f"Episode {episode} | Reward: {episode_reward:.2f}")

    # 在回合结束后记录数据
    with open(log_file, 'a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([episode, episode_reward, t+1])

    # 每100回合保存一次中间结果
    if episode > 0 and episode % 100 == 0:
        output_dir = Path(args.output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)

        actor_path = output_dir / f"td3_actor_{episode}.pth"
        critic_path = output_dir / f"td3_critic_{episode}.pth"

        torch.save(agent.actor.state_dict(), actor_path)
        torch.save(agent.critic.state_dict(), critic_path)
        print(f"[DEBUG] Saved intermediate models to {actor_path}")

# 训练完成后保存最终模型参数
os.makedirs(args.output_dir, exist_ok=True)
torch.save(agent.actor.state_dict(), os.path.join(args.output_dir, "td3_actor_final.pth"))
torch.save(agent.critic.state_dict(), os.path.join(args.output_dir, "td3_critic_final.pth"))
print(f"[DEBUG] Training complete. Final models saved to: {args.output_dir}")

Episode 0 | Reward: -79.58
Episode 1 | Reward: -63.93
Episode 2 | Reward: 31.32
Episode 3 | Reward: -84.98
Episode 4 | Reward: -95.40
Episode 5 | Reward: -155.05
Episode 6 | Reward: -124.89
Episode 7 | Reward: -110.45
Episode 8 | Reward: -252.01
Episode 9 | Reward: -161.55
Episode 10 | Reward: -46.34
Episode 11 | Reward: 39.33
Episode 12 | Reward: 166.64
Episode 13 | Reward: -200.00
Episode 14 | Reward: -200.00
Episode 15 | Reward: -190.60
Episode 16 | Reward: -147.60
Episode 17 | Reward: 148.41
Episode 18 | Reward: -2.98
Episode 19 | Reward: -196.67
Episode 20 | Reward: 25.79
Episode 21 | Reward: -60.82
Episode 22 | Reward: -61.01
Episode 23 | Reward: -72.91
Episode 24 | Reward: -7.83
Episode 25 | Reward: -198.16
Episode 26 | Reward: 148.46
Episode 27 | Reward: 39.76
Episode 28 | Reward: -22.04
Episode 29 | Reward: -200.00
Episode 30 | Reward: 58.00
Episode 31 | Reward: 163.43
Episode 32 | Reward: -0.61
Episode 33 | Reward: -100.24
Episode 34 | Reward: -140.60
Episode 35 | Reward: -20

[DEBUG] Output directory: ./results_drone
[DEBUG] Directory exists: True
Episode 0 | Reward: 23.65
Episode 1 | Reward: -76.91
Episode 2 | Reward: -20.89
Episode 3 | Reward: -71.52
Episode 4 | Reward: -217.86
Episode 5 | Reward: 11.44
Episode 6 | Reward: -160.17
Episode 7 | Reward: -130.46
Episode 8 | Reward: -171.68
Episode 9 | Reward: -15.65
Episode 10 | Reward: -137.91
Episode 11 | Reward: -187.79
Episode 12 | Reward: -173.06
Episode 13 | Reward: -260.88
Episode 14 | Reward: -233.77
Episode 15 | Reward: -262.37
Episode 16 | Reward: 23.98
Episode 17 | Reward: -38.41
Episode 18 | Reward: -127.07
Episode 19 | Reward: -230.32
Episode 20 | Reward: -165.52
Episode 21 | Reward: -58.09
Episode 22 | Reward: -173.33
Episode 23 | Reward: -42.46
Episode 24 | Reward: -80.67
Episode 25 | Reward: -127.74
Episode 26 | Reward: -142.00
Episode 27 | Reward: -119.72
Episode 28 | Reward: -125.75
Episode 29 | Reward: -133.37
Episode 30 | Reward: 176.34
Episode 31 | Reward: -145.63
Episode 32 | Reward: -54

In [ ]:
#analysis.py，也就是分析训练结果
import pandas as pd
import matplotlib.pyplot as plt
import os
import argparse
import sys
import glob
import seaborn as sns

# 解析命令行参数
parser = argparse.ArgumentParser()
parser.add_argument('--runs_dir', default='results_drone_Fix_noise', help='Directory containing log csv files')
parser.add_argument('--output_dir', default='analysis_results_Fix_noise', help='Directory to save analysis results')

# 适配 Colab/Jupyter 环境
if 'ipykernel' in sys.modules:
    args = parser.parse_args(args=['--runs_dir', 'results_drone_Fix_noise', '--output_dir', 'analysis_results_Fix_noise'])
else:
    args = parser.parse_args()

# 创建输出目录
os.makedirs(args.output_dir, exist_ok=True)

# === 修改的核心部分：直接读取目录下所有的 csv 日志文件 ===
all_data = []
# 查找所有的 csv 文件
csv_pattern = os.path.join(args.runs_dir, '*.csv')
csv_files = glob.glob(csv_pattern)

for file_path in csv_files:
    # 排除掉之前生成的 combined 汇总文件，防止重复读取
    if 'combined' in file_path:
        continue

    df = pd.read_csv(file_path)
    # 用文件名（比如 training_log_xxx.csv）作为这一次运行的 ID
    df['run_id'] = os.path.basename(file_path)
    all_data.append(df)

if not all_data:
    print(f"错误: 在 {args.runs_dir} 目录下没有找到任何 CSV 日志文件！请检查训练是否成功生成了日志。")
    sys.exit()

# # 合并所有运行数据
# combined_df = pd.concat(all_data, ignore_index=True)

# # 计算基本统计量
# mean_reward = combined_df.groupby('episode')['reward'].mean().reset_index()
# std_reward = combined_df.groupby('episode')['reward'].std().reset_index()

# # 数据有效性检查
# print(f"调试信息: 找到 {len(all_data)} 个日志文件")
# 合并所有运行数据
combined_df = pd.concat(all_data, ignore_index=True)

# ================= 新增修改脏数据清洗与转换 =================
# 1. 强制将这两列转换为纯数字格式 (float)，遇到乱码或字符串自动变成 NaN
combined_df['reward'] = pd.to_numeric(combined_df['reward'], errors='coerce')
combined_df['episode'] = pd.to_numeric(combined_df['episode'], errors='coerce')

# 2. 剔除那些转换失败的空行
combined_df = combined_df.dropna(subset=['reward', 'episode'])
# ======================================================================

# 计算基本统计量
mean_reward = combined_df.groupby('episode')['reward'].mean().reset_index()
std_reward = combined_df.groupby('episode')['reward'].std().reset_index()

# ================= 新增：处理单文件导致的 NaN 标准差 =================
# 如果只有一个文件，std() 会返回 NaN。我们将其填充为 0.0，避免画图崩溃
std_reward['reward'] = std_reward['reward'].fillna(0.0)
# ======================================================================

# 数据有效性检查
print(f"调试信息: 找到 {len(all_data)} 个日志文件")

print(f"调试信息: 唯一episode数量: {len(combined_df['episode'].unique())}")
print(f"调试信息: 输出目录: {args.output_dir}")

if len(combined_df['episode'].unique()) < 2:
    print("警告: 检测到数据中仅包含单个episode，无法生成有效曲线")
    # 生成单数据点占位图像
    plt.figure(figsize=(12, 8))
    episode_value = combined_df['episode'].unique()[0]
    reward_mean = combined_df['reward'].mean()
    plt.scatter(episode_value, reward_mean, color='red')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title('Insufficient Data: Only Single Episode Available')
    plt.grid(True)
    plt.savefig(os.path.join(args.output_dir, 'all_runs_reward_curve.png'))
    plt.close()
else:
    # 绘制所有运行的奖励曲线
    plt.figure(figsize=(12, 8))
    sns.lineplot(data=combined_df, x='episode', y='reward', hue='run_id', alpha=0.5)
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title('Reward Curves for All Training Runs')
    plt.grid(True)
    # 把图例移到图外边，防止遮挡曲线
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'all_runs_reward_curve.png'))
    plt.close()

    # 绘制平均奖励曲线
    plt.figure(figsize=(12, 8))
    plt.plot(mean_reward['episode'], mean_reward['reward'], label='Mean Reward', color='blue')
    plt.fill_between(mean_reward['episode'],
                     mean_reward['reward'] - std_reward['reward'],
                     mean_reward['reward'] + std_reward['reward'],
                     alpha=0.2, color='blue', label='Std Dev')
    plt.xlabel('Episode')
    plt.ylabel('Reward')
    plt.title('Average Reward with Standard Deviation')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(args.output_dir, 'average_reward_curve.png'))
    plt.close()

# 绘制奖励分布箱线图 (如果 Episode 很多，箱线图会很密，这里做了每 10 个 episode 采样)
plt.figure(figsize=(12, 8))
sampled_df = combined_df[combined_df['episode'] % 10 == 0] # 只画出 0, 10, 20... 的箱线图
if not sampled_df.empty:
    sns.boxplot(data=sampled_df, x='episode', y='reward')
    plt.xlabel('Episode (Sampled every 10)')
    plt.ylabel('Reward Distribution')
    plt.title('Reward Distribution Across Episodes')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(args.output_dir, 'reward_boxplot.png'))
plt.close()

# 输出总体统计信息
print('所有运行的总体统计信息：')
print(combined_df[['reward', 'steps']].describe())

# 保存合并后的数据
combined_df.to_csv(os.path.join(args.output_dir, 'combined_training_log.csv'), index=False)
print(f"✅ 所有图表和数据已成功保存至: {args.output_dir}")

调试信息: 找到 1 个日志文件
调试信息: 唯一episode数量: 5000
调试信息: 输出目录: analysis_results
所有运行的总体统计信息：
            reward        steps
count  5000.000000  5000.000000
mean     72.158617   119.308800
std     108.401897    81.480609
min    -366.974276     1.000000
25%      21.061858    34.000000
50%      74.887849   138.500000
75%     160.835265   200.000000
max     279.591542   200.000000
✅ 所有图表和数据已成功保存至: analysis_results


In [ ]:
# 压缩并下载训练结果文件夹，包含模型和分析结果。注意路径要根据实际情况调整。
# 1. 使用 zip 命令将整个文件夹压缩为 .zip 文件
# -r 表示递归压缩（包含文件夹内的所有子文件和子文件夹）
!zip -r /content/results_drone_Fix_noise.zip /content/results_drone_Fix_noise

!zip -r /content/analysis_results_Fix_noise.zip /content/analysis_results_Fix_noise
# 2. 引入 Colab 的文件下载模块
from google.colab import files

# 3. 触发浏览器下载
files.download('/content/results_drone_Fix_noise.zip')
files.download('/content/analysis_results_Fix_noise.zip')

  adding: content/results_drone/ (stored 0%)
  adding: content/results_drone/td3_actor_400.pth (deflated 7%)
  adding: content/results_drone/td3_critic_4300.pth (deflated 6%)
  adding: content/results_drone/td3_actor_4000.pth (deflated 7%)
  adding: content/results_drone/td3_actor_3100.pth (deflated 7%)
  adding: content/results_drone/td3_actor_3200.pth (deflated 7%)
  adding: content/results_drone/td3_actor_3800.pth (deflated 7%)
  adding: content/results_drone/td3_actor_600.pth (deflated 7%)
  adding: content/results_drone/td3_critic_1300.pth (deflated 6%)
  adding: content/results_drone/td3_critic_4200.pth (deflated 6%)
  adding: content/results_drone/td3_actor_1300.pth (deflated 7%)
  adding: content/results_drone/td3_actor_1200.pth (deflated 7%)
  adding: content/results_drone/td3_critic_4400.pth (deflated 6%)
  adding: content/results_drone/td3_critic_4800.pth (deflated 6%)
  adding: content/results_drone/td3_critic_1500.pth (deflated 6%)
  adding: content/results_drone/td3_actor

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# 三维仿真前置:导入权重文件
import os
import torch

# 确保设备设置正确
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
directory = "results_drone_Fix_noise" # 你的存档文件夹

print(f"🔍 正在扫描文件夹 {directory} ...")
if not os.path.exists(directory):
    print("❌ 文件夹不存在，请检查路径！")
else:
    # 1. 寻找所有的 .pth 权重文件
    files = os.listdir(directory)
    pth_files = [f for f in files if f.endswith('.pth')]
    print(f"找到的权重文件: {pth_files}")

    if len(pth_files) == 0:
        print("❌ 没有找到任何 .pth 文件！请检查你的模型是否存到了其他地方。")
    else:
        # 2. 优先寻找名字里带 'actor' 的文件，如果没有就随便拿第一个
        actor_file = next((f for f in pth_files if 'actor' in f.lower()), pth_files[0])
        file_path = os.path.join(directory, actor_file)
        
        print(f"🚀 准备将记忆注入大脑: {file_path}")
        
        # 3. 使用 PyTorch 原生方法强行加载权重
        try:
            # 读取文件，并确保它被放到正确的设备(GPU/CPU)上
            state_dict = torch.load(file_path, map_location=device)
            agent.actor.load_state_dict(state_dict)
            print("✅ 成功加载满级大脑！现在可以去跑画 3D 录像的代码了！")
            
        except RuntimeError as e:
            print(f"❌ 加载失败！网络维度可能不匹配（比如存档是 6 维的，但你现在用的是 14 维的网络）。")
            print(f"详细报错: {e}")
        except Exception as e:
            print(f"❌ 发生未知错误: {e}")

🔍 正在扫描文件夹 results_drone ...
找到的权重文件: ['td3_actor_3600.pth', 'td3_critic_4600.pth', 'td3_critic_200.pth', 'td3_actor_500.pth', 'td3_critic_1000.pth', 'td3_actor_2000.pth', 'td3_actor_800.pth', 'td3_actor_1700.pth', 'td3_critic_2700.pth', 'td3_critic_3100.pth', 'td3_actor_4100.pth', 'td3_critic_final.pth', 'td3_actor_700.pth', 'td3_actor_3400.pth', 'td3_critic_4400.pth', 'td3_actor_2200.pth', 'td3_critic_1200.pth', 'td3_actor_3900.pth', 'td3_critic_4900.pth', 'td3_critic_2500.pth', 'td3_actor_1500.pth', 'td3_critic_3300.pth', 'td3_actor_4300.pth', 'td3_actor_1800.pth', 'td3_critic_2800.pth', 'td3_actor_2400.pth', 'td3_critic_1400.pth', 'td3_critic_1900.pth', 'td3_actor_2900.pth', 'td3_actor_100.pth', 'td3_critic_600.pth', 'td3_actor_3200.pth', 'td3_critic_4200.pth', 'td3_critic_3500.pth', 'td3_actor_4500.pth', 'td3_critic_3800.pth', 'td3_actor_4800.pth', 'td3_critic_2300.pth', 'td3_actor_1300.pth', 'td3_critic_1600.pth', 'td3_actor_2600.pth', 'td3_critic_900.pth', 'td3_actor_3000.pth', '

In [22]:
# 三维仿真: 纯净测试
import numpy as np
import torch

# 1. 临时关闭探索噪声，进行“纯净测试”
# 确保你已经实例化了 env 和 agent，并且加载了最好的模型权重
original_noise = agent.policy_noise
agent.policy_noise = 0.0  # 设为0，让它完全按照学到的最优策略飞行

# 2. 初始化环境，准备“黑匣子”记录器
state = env.reset()
done = False

# 记录核心数据
trajectory = []                # 记录飞行的轨迹坐标
start_pos = env.position.copy() # 记录起点
target_pos = env.target.copy()  # 记录终点
obstacles_data = env.obstacles.copy() # 记录障碍物的坐标和半径

# 记录起点
trajectory.append(start_pos.copy())

# 3. 开始闭环飞行测试
print("🚁 无人机起飞，开始纯净飞行测试...")
step_count = 0

while not done:
    # 获取动作 (此时没有随机噪声)
    action = agent.select_action(state)
    
    # 执行动作
    state, reward, done, info = env.step(action)
    
    # 记录当前位置
    trajectory.append(env.position.copy())
    step_count += 1

# 转换为 NumPy 数组方便后续画图
trajectory = np.array(trajectory)

# 恢复训练时的噪声设置（好习惯）
agent.policy_noise = original_noise

# 打印最终结果
final_distance = np.linalg.norm(env.target - env.position)
print(f"✅ 飞行结束！共耗时 {step_count} 步。")
if final_distance < 1.5:
    print(f"🎯 成功到达终点！距目标仅 {final_distance:.2f} 米。")
else:
    print(f"💥 发生碰撞或超时。距目标还有 {final_distance:.2f} 米。")

🚁 无人机起飞，开始纯净飞行测试...
✅ 飞行结束！共耗时 46 步。
🎯 成功到达终点！距目标仅 1.39 米。


🎯 三维仿真依赖包👇

In [ ]:
!pip install plotly
!pip install --upgrade nbformat

In [23]:
#三维仿真绘图
import plotly.graph_objects as go
import numpy as np

# 1. 初始化 3D 画布
fig = go.Figure()

# 2. 画起点和终点
fig.add_trace(go.Scatter3d(
    x=[start_pos[0]], y=[start_pos[1]], z=[start_pos[2]],
    mode='markers', marker=dict(size=6, color='blue'), name='起点 (Start)'
))
fig.add_trace(go.Scatter3d(
    x=[target_pos[0]], y=[target_pos[1]], z=[target_pos[2]],
    mode='markers', marker=dict(size=8, color='green', symbol='diamond'), name='终点 (Target)'
))

# 3. 画无人机的飞行轨迹线
fig.add_trace(go.Scatter3d(
    x=trajectory[:, 0], y=trajectory[:, 1], z=trajectory[:, 2],
    mode='lines+markers',
    line=dict(color='orange', width=6),
    marker=dict(size=3, color='orange'),
    name='无人机轨迹'
))

# 4. 数学建模：生成 3D 球体表面的网格数据
def create_sphere_mesh(center, radius, resolution=20):
    u = np.linspace(0, 2 * np.pi, resolution)
    v = np.linspace(0, np.pi, resolution)
    x = center[0] + radius * np.outer(np.cos(u), np.sin(v))
    y = center[1] + radius * np.outer(np.sin(u), np.sin(v))
    z = center[2] + radius * np.outer(np.ones(np.size(u)), np.cos(v))
    return x, y, z

# 5. 把所有障碍物画到图中
for i, obs in enumerate(obstacles_data):
    cx, cy, cz = obs['pos']
    r = obs['radius']
    sx, sy, sz = create_sphere_mesh((cx, cy, cz), r)
    
    fig.add_trace(go.Surface(
        x=sx, y=sy, z=sz,
        colorscale='Reds',      # 红色代表危险
        opacity=0.4,            # 半透明，避免挡住视线
        showscale=False,
        name=f'障碍物 {i+1}'
    ))

# 6. 设置画布的视角和比例
fig.update_layout(
    scene=dict(
        xaxis_title='X (米)',
        yaxis_title='Y (米)',
        zaxis_title='Z (米)',
        # 【关键设置】 aspectmode='data' 强制 XYZ 比例为 1:1:1，确保球体不会变成椭圆
        aspectmode='data' 
    ),
    title="无人机 3D 避障航线分析视图",
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0.02, y=0.98)
)

# 7. 显示交互图表
fig.show()
# fig.show(renderer="notebook")#适合大多数网页版 Jupyter
# fig.show(renderer="iframe")#适合经典的 Jupyter Notebook
# fig.show(renderer="vscode")#VS Code 的网页版
fig.write_html("drone_3d_flight_Fix_noise.html")#直接导出为独立 HTML 网页
print("3D仿真图已保存为 drone_3d_flight_Fix_noise.html！")

3D仿真图已保存为 drone_3d_flight.html！


In [24]:
#生成 3D 飞行录像代码
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
# 导入 3D 绘图模块
from mpl_toolkits.mplot3d import Axes3D

# 1. 初始化画布和 3D 坐标轴
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 2. 画出起点（蓝色）和终点（绿色星号）
ax.scatter(*start_pos, color='blue', s=100, label='Start', edgecolors='black')
ax.scatter(*target_pos, color='green', s=150, marker='*', label='Target', edgecolors='black')

# 3. 绘制所有的障碍物（红色半透明网格球体）
for obs in obstacles_data:
    cx, cy, cz = obs['pos']
    r = obs['radius']
    u = np.linspace(0, 2 * np.pi, 20)
    v = np.linspace(0, np.pi, 20)
    x = cx + r * np.outer(np.cos(u), np.sin(v))
    y = cy + r * np.outer(np.sin(u), np.sin(v))
    z = cz + r * np.outer(np.ones(np.size(u)), np.cos(v))
    # 使用 wireframe 画网格，避免遮挡视线
    ax.plot_wireframe(x, y, z, color='red', alpha=0.2)

# 4. 固定坐标轴的显示范围
# 之前你的代码里撞墙边界是 15，所以我们把视野固定在 [-15, 15] 
# 这样可以防止动画播放时镜头跟着无人机乱晃
ax.set_xlim([-15, 15])
ax.set_ylim([-15, 15])
ax.set_zlim([-15, 15])
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
ax.set_title('Drone 3D Flight Animation')
ax.legend()

# 5. 初始化动画中的“动态元素”：飞行轨迹线 和 无人机当前位置点
trail_line, = ax.plot([], [], [], color='orange', linewidth=2, label='Trajectory')
drone_point, = ax.plot([], [], [], 'o', color='red', markersize=8, label='Drone')

# 动画初始化函数
def init():
    trail_line.set_data([], [])
    trail_line.set_3d_properties([])
    drone_point.set_data([], [])
    drone_point.set_3d_properties([])
    return trail_line, drone_point

# 动画的每一帧更新函数
def update(frame):
    # 获取从第 0 帧到当前帧的所有历史轨迹
    current_traj = trajectory[:frame+1]
    
    # 更新轨迹线
    trail_line.set_data(current_traj[:, 0], current_traj[:, 1])
    trail_line.set_3d_properties(current_traj[:, 2])
    
    # 更新无人机当前的红点位置
    current_pos = trajectory[frame]
    drone_point.set_data([current_pos[0]], [current_pos[1]])
    drone_point.set_3d_properties([current_pos[2]])
    
    return trail_line, drone_point

print("⏳ 正在全力渲染 3D 逐帧动画，请稍候（可能需要几十秒）...")

# 6. 生成动画！(interval=50 表示每帧间隔 50 毫秒，即 20 FPS)
# frames 取轨迹的总步数
ani = animation.FuncAnimation(fig, update, frames=len(trajectory),
                              init_func=init, blit=False, interval=50)

# 7. 关闭静态多余的图表显示，并将动画转为 HTML5 交互式播放器
plt.close()
HTML(ani.to_jshtml())

⏳ 正在全力渲染 3D 逐帧动画，请稍候（可能需要几十秒）...
